In [ ]:
import numpy as np
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
# Sample Dataset (or use pd.read_csv for your own dataset)
# df = pd.read_csv('your_data.csv')
# X = df['text_column'].tolist()
# y = df['label_column'].tolist()

X = ["great product", "best service", "love it", "excellent", "bad", "terrible", "waste", "poor"]
y = [1, 1, 1, 1, 0, 0, 0, 0]

In [ ]:
def preprocess(text):
    sw = set(stopwords.words('english'))
    return [w for w in word_tokenize(text.lower()) if w.isalnum() and w not in sw]

In [ ]:
def one_hot(doc, vocab):
    vec = np.zeros(len(vocab))
    for w in doc:
        if w in vocab: vec[vocab.index(w)] = 1
    return vec

In [ ]:
processed = [preprocess(s) for s in X]
vocab = list(set(w for d in processed for w in d))
X_train = np.array([one_hot(d, vocab) for d in processed])

In [ ]:
class SimpleDecisionTree:
    def __init__(self, max_depth=2):
        self.max_depth = max_depth
    
    def fit(self, X, y):
        self.tree = self._build(X, y, 0)
    
    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(np.unique(y)) == 1:
            return np.bincount(y).argmax()
        best = (float('inf'), None)
        for f in range(X.shape[1]):
            for t in np.unique(X[:, f]):
                l, r = y[X[:, f] <= t], y[X[:, f] > t]
                gini = lambda y: 1 - sum((np.bincount(y)/len(y))**2)
                g = (len(l)*gini(l) + len(r)*gini(r))/len(y)
                if g < best[0]:
                    best = (g, (f, t))
        if best[1] is None: return np.bincount(y).argmax()
        f, t = best[1]
        l_idx = X[:, f] <= t
        return (f, t, self._build(X[l_idx], y[l_idx], depth+1), self._build(X[~l_idx], y[~l_idx], depth+1))
    
    def predict(self, X):
        preds = []
        for x in X:
            tree = self.tree
            while isinstance(tree, tuple):
                f, t, l, r = tree
                tree = l if x[f] <= t else r
            preds.append(tree)
        return preds

In [ ]:
class SimpleRandomForest:
    def __init__(self, n_trees=10):
        self.trees = [SimpleDecisionTree() for _ in range(n_trees)]
    
    def fit(self, X, y):
        for t in self.trees:
            idx = np.random.choice(len(X), len(X))
            t.fit(X[idx], y[idx])
    
    def predict(self, X):
        all_preds = np.array([t.predict(X) for t in self.trees]).T
        return [np.bincount(p).argmax() for p in all_preds]

In [ ]:
model = SimpleRandomForest()
model.fit(X_train, y)

In [ ]:
test = one_hot(preprocess("great service"), vocab)
print("Prediction:", model.predict([test])[0])